In [1]:
# env 파일 읽어오기

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

MODEL = 'gpt-6-luna'

#### 구조화된 출력
- JSON 형태로 받는게 가장 편함
- 결과값을 JSON 형태로 받기 위해 JSON SCHEMA or Pydantic 사용

In [2]:
# 사용자의 자기소개에서 이름과 나이를 추출
from openai import OpenAI
import json

client = OpenAI()

response = client.responses.create(
    model=MODEL,
    input="안녕하세요. 저는 김민수이고 23살입니다.",
    text={
        "format":{
            "type":"json_schema",
            "name":"user_info",
            "strict":True,      # 모든 객체의 필드를 required에 포함
            "schema":{
                "type":"object",
                "properties":{
                    "name":{
                        "type":"string"
                    },
                    "age":{
                        "type":"integer"
                    }
                },
                "required":["name","age"],
                "additionalProperties":False     # 정의하지 않은 필드의 반환 제한
            }
        }
    }
)

result = json.loads(response.output_text)
print(result)



{'name': '김민수', 'age': 23}


In [3]:
# 상품 정보에서 문자열, 숫자, 불리언
# {'name':'무선키보드', 'price':45000, 'rating':4.7, 'in_stock':True}
response = client.responses.create(
    model=MODEL,
    input="상품명은 무선 키보드이고, 가격은 45000원입니다. 재고는 현재 있으며, 평점은 4.7점입니다.",
    text={
        "format":{
            "type":"json_schema",
            "name":"product_info",
            "strict":True,      # 모든 객체의 필드를 required에 포함
            "schema":{
                "type":"object",
                "properties":{
                    "name":{
                        "type":"string"
                    },
                    "price":{
                        "type":"integer"
                    },
                    "rating":{
                        "type":"number"      # 정수, 실수 다 허용
                    },
                    "in_stock":{
                        "type":"boolean"
                    }
                },
                "required":["name","price", "rating", "in_stock"],
                "additionalProperties":False     # 정의하지 않은 필드의 반환 제한
            }
        }
    }
)

result = json.loads(response.output_text)
print(result)


{'name': '무선 키보드', 'price': 45000, 'rating': 4.7, 'in_stock': True}


In [4]:
# 고객문의를 분석하여 문의 유형과 중요도로 응답
# {'category':'delivery', 'priority':'high', 'summary':'요청사항}


response = client.responses.create(
    model=MODEL,
    input="결제는 완료했는데 상품이 배송되지 않고 있습니다. 빠르게 확인해주세요.",
    text={
        "format":{
            "type":"json_schema",
            "name":"customer_inquiry",
            "strict":True,      # 모든 객체의 필드를 required에 포함
            "schema":{
                "type":"object",
                "properties":{
                    "category":{
                        "type":"string",
                        "enum":["payment","delivery","refund","etc"]
                    },
                    "delivery":{
                        "type":"integer",
                    },
                    "priority":{
                        "type":"string",      
                        "enum":["high","medium","low"]
                    },
                    "summary":{
                        "type":"string"
                    }
                },
                "required":["category","delivery", "priority", "summary"],
                "additionalProperties":False     # 정의하지 않은 필드의 반환 제한
            }
        }
    }
)

result = json.loads(response.output_text)
print(result)

{'category': 'delivery', 'delivery': 0, 'priority': 'high', 'summary': '결제는 완료했으나 상품이 배송되지 않아 빠른 확인을 요청합니다.'}


In [5]:
input = """
제품을 구매한지 일주일이 지났는데 아직 배송되지 않았습니다. 
고객센터에 여러 번 문의했지만 답변도 받지 못했습니다.
환불을 요청합니다.
"""

# {sentiment:negative, intent:refund, keywords:[배송재연, 고객센터 미응답, 환불 요청], summary:'~~~}
# sentiment: positive, neutral, negative
# intent : delivery, refund, exchange, complaint, etc
# keywords

response = client.responses.create(
    model=MODEL,
    input=input,
    text={
        "format":{
            "type":"json_schema",
            "name":"sentiment_analysis",
            "strict":True,      # 모든 객체의 필드를 required에 포함
            "schema":{
                "type":"object",
                "properties":{
                    "sentiment":{
                        "type":"string",
                        "enum":["negative","positive"]
                    },
                    "intent":{
                        "type":"string",
                        "enum":["delivery","refund","exchange","complaint","etc"]
                    },
                    "keywords":{
                        "type":"array",
                        "items":{"type":"string"}
                    }
                },
                "required":["sentiment","intent", "keywords"],
                "additionalProperties":False     # 정의하지 않은 필드의 반환 제한
            }
        }
    }
)

result = json.loads(response.output_text)
print(result)


{'sentiment': 'negative', 'intent': 'refund', 'keywords': ['배송 지연', '일주일 미배송', '고객센터 답변 없음', '환불 요청']}


In [6]:
print(json.dumps(result, ensure_ascii=False, indent=4))

{
    "sentiment": "negative",
    "intent": "refund",
    "keywords": [
        "배송 지연",
        "일주일 미배송",
        "고객센터 답변 없음",
        "환불 요청"
    ]
}


In [7]:
# 계약서 내용을 분석하여 계약 당사자, 계약 기간, 계약 금액 등을 추출
# {contract_date:2026-03-01, parties:[주식회사 ABC, 김철수], period:{start_date:2026-03-01, end_date:2027-02-28}, amount:5000000}
input="""
계약서 내용:

계약 당사자는 주식회사 ABC와 김철수이다. 
계약 체결일은 2026년 3월 1일이다.
계약 기간은 2026년 3월 1일부터 2027년 2월 28일까지이다. 
계약 금액은 5,000,000이다.
"""

response = client.responses.create(
    model=MODEL,
    input=input,
    text={
        "format":{
            "type":"json_schema",
            "name":"contract_info",
            "strict":True,      # 모든 객체의 필드를 required에 포함
            "schema":{
                "type":"object",
                "properties":{
                    "contract_date":{
                        "type":"string",
                    },
                    "parties":{
                        "type":"array",
                        "items":{
                            "type":"string"
                        }
                    },
                    "periods":{
                        "type":"object",
                        "properties":{
                            "start_date":{
                                "type":"string"
                            },
                            "end_date":{
                                "type":"string"
                            }
                        },
                        "required":["start_date","end_date"],
                        "additionalProperties":False
                    },
                    "amount":{
                        "type":"integer"
                    }
                },
                "required":["contract_date","parties", "periods", "amount"],
                "additionalProperties":False     # 정의하지 않은 필드의 반환 제한
            }
        }
    }
)

result = json.loads(response.output_text)
print(result)

{'contract_date': '2026년 3월 1일', 'parties': ['주식회사 ABC', '김철수'], 'periods': {'start_date': '2026년 3월 1일', 'end_date': '2027년 2월 28일'}, 'amount': 5000000}


### Pydantic

In [9]:
# 이벤트 정보 추출 - 이벤트명, 날짜, 참가자
# 앨리스와 밥이 금요일에 과학 박람회에 갈 예정입니다.

from pydantic import BaseModel

client = OpenAI()

class CalendarEvent(BaseModel):
    name:str
    date:str
    participants:list[str]

input = "앨리스와 밥이 금요일에 과학 박람회에 갈 예정입니다."

response = client.responses.parse(
    model=MODEL,
    instructions='이벤트 정보를 한국어로 추출하세요.',
    input=input,
    text_format=CalendarEvent
)

print(response.output_parsed)

name='과학 박람회' date='금요일' participants=['앨리스', '밥']


In [11]:
input = "무선 키보드이며, 가격은 450000원이고, 전자제품이며 재고가 있습니다."

class ProductInfo(BaseModel):
    name:str
    price:int
    category:str
    in_stock:bool


response = client.responses.parse(
    model=MODEL,
    input=input,
    text_format=ProductInfo
)

print(response.output_parsed)

name='무선 키보드' price=450000 category='전자제품' in_stock=True


In [13]:

class Step(BaseModel):
    explain:str
    output:str


class MathReasoning(BaseModel):
    steps:list[Step]
    final_answer:str


response = client.responses.parse(
    model=MODEL,
    input=[
        {"role":"developer", "content":"당신은 친절한 수학 선생님입니다. 사용자가 문제를 단계별로 이해하고 풀 수 있도록 안내하세요."},
        {"role":"user", "content":"8x + 7 = -23 방정식을 어떻게 풀 수 있나요?"}
    ],
    text_format=MathReasoning
)

print(response.output_parsed)

steps=[Step(explain='먼저 양변에서 7을 빼서 x가 포함된 항만 남깁니다.', output='8x + 7 - 7 = -23 - 7 → 8x = -30'), Step(explain='이제 양변을 8로 나누어 x를 구합니다.', output='x = -30 ÷ 8 = -15/4'), Step(explain='소수로 나타내면 -15/4 = -3.75입니다.', output='따라서 x = -15/4 (또는 -3.75)')] final_answer='x = -15/4 (또는 -3.75)'


In [17]:
# 주문번호 A-1923 아직도 안 왔어요. 3만9천원 환불해 주세요. 진짜 화나네요.
# category, priority, sentiment_score(-1 ~ 1), entities, refund_amount, 
# entities : kind, value => order_id, A-1293

from typing import Literal, Optional
from pydantic import Field

class Entity(BaseModel):
    kind:Literal['order_id', 'product', 'date']
    value:str


class Ticket(BaseModel):
    category:Literal['배송','환불','결제','기술지원','기타']
    priority:Literal['low','medium','high']
    sentiment_score:float = Field(description="-1(매우 부정) ~ 1(매우 긍정)")
    entities:list[Entity]
    refund_amount:Optional[int]
    summary:str

def classify(text:str) -> Ticket | None:
    response = client.responses.parse(
        model=MODEL,
        input=[
            {"role":"system", "content":"당신은 쇼핑몰 회사의 직원입니다. 고객 문의를 분류하고 핵심 엔티티를 추출하세요."},
            {"role":"user", "content":text}
        ],
        text_format=Ticket
    )

    print(response.output_parsed)

classify("주문번호 A-1923 아직도 안 왔어요. 3만9천원 환불해 주세요. 진짜 화나네요.")


category='배송' priority='high' sentiment_score=-0.9 entities=[Entity(kind='order_id', value='A-1923')] refund_amount=39000 summary='주문번호 A-1923 상품이 아직 도착하지 않아 3만 9천 원 환불을 요청했으며, 고객이 매우 화가 난 상태입니다.'


In [ ]:
input = """
계약서 내용:

계약 당사자는 주식회사 ABC와 김철수이다.
계약 체결일은 2026년 3월 1일이다.
계약 기간은 2026년 3월 1일부터 2027년 2월 28일까지이다.
계약 금액은 5,000,000이다.
"""


class Period(BaseModel):
    start_date: str
    end_date: str


class ContractInfo(BaseModel):
    contract_date: str
    parties: list[str]
    period: Period      
    amount: int


response = client.responses.parse(
    model=MODEL,
    instructions="계약서에서 정보를 추출하세요. 날짜는 YYYY-MM-DD 형식으로 작성하세요.",
    input=input,
    text_format=ContractInfo     
)

result = response.output_parsed  # json.loads 없이 바로 ContractInfo 객체
print(result)


contract_date='2026-03-01' parties=['주식회사 ABC', '김철수'] period=Period(start_date='2026-03-01', end_date='2027-02-28') amount=5000000
